# Your First PINN: Solving the Heat Equation

> **Repository:** [PINNs-RL-PDE](https://github.com/josegarciav/PINNs-RL-PDE) &nbsp;|&nbsp; **Package:** `pinnrl` &nbsp;|&nbsp; **Estimated run time:** ~2 minutes on CPU

This notebook gives you a hands-on introduction to **Physics-Informed Neural Networks (PINNs)**. 
You will solve the 1-D heat equation from scratch — no black-box trainer — so every step is visible.

---

## What is a PINN?

A Physics-Informed Neural Network is a standard neural network whose **loss function encodes the physics** of a differential equation:

```
                    ┌────────────────────────────────────────┐
                    │           Neural Network               │
  (x, t) ──────────►   u_θ(x,t)  ──► predictions           │
                    └───────────────────┬────────────────────┘
                                        │  automatic differentiation
                                        ▼
             ┌──────────────┬──────────────────┬───────────────┐
             │  PDE Loss    │  Boundary Loss   │ Initial Loss  │
             │  L_pde       │  L_bc            │ L_ic          │
             └──────┬───────┴────────┬─────────┴──────┬────────┘
                    └────────────────▼────────────────┘
                             Total Loss L = L_pde + L_bc + L_ic
                                        │
                                   back-prop
                                   optimizer
```

The network **learns the solution** u(x, t) by minimising all three terms simultaneously.

## 1  The Physics: 1-D Heat Equation

We want to solve:

$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}, \quad x\in[0,1],\; t\in[0,1]$$

with:
- **Initial condition:** $u(x, 0) = \sin(2\pi x)$
- **Periodic boundary conditions:** $u(0,t) = u(1,t)$, $u_x(0,t)=u_x(1,t)$
- **Thermal diffusivity:** $\alpha = 0.1$

The **exact analytical solution** is:

$$u(x,t) = \exp\!\left(-\alpha\,(2\pi)^2\,t\right)\,\sin(2\pi x)$$

This is what we want the network to learn.

## 2  Setup & Imports

In [1]:
import sys, os

# ── Make sure the repo root is on the Python path ──────────────────────────
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time

# Use a clean, readable style
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

device = torch.device("cpu")   # CPU is fine for this 50-epoch demo
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")
print(f"Repo root: {REPO_ROOT}")

PyTorch  : 2.10.0
Device   : cpu
Repo root: /Users/josegarcia/Documents/GitHub/PINNs-RL-PDE


## 3  Load Config and Build the PDE

We use `pinnrl`'s `Config` class (reads `config.yaml`) and `HeatEquation` from the PDE library.
Then we override the key physics parameters in-memory so the notebook is self-contained.

In [2]:
from pinnrl.config import Config, ModelConfig
from pinnrl.pdes.pde_base import PDEConfig
from pinnrl.pdes.heat_equation import HeatEquation

# ── Physics parameters ──────────────────────────────────────────────────────
ALPHA = 0.1       # thermal diffusivity
FREQ  = 1.0       # spatial frequency (sin(2*pi*freq*x))
X_MIN, X_MAX = 0.0, 1.0
T_MIN, T_MAX = 0.0, 1.0

# Build a PDEConfig that matches the HeatEquation expectations
pde_cfg = PDEConfig(
    name="Heat Equation (notebook demo)",
    domain=[[X_MIN, X_MAX]],          # list of [min, max] per spatial dim
    time_domain=[T_MIN, T_MAX],
    parameters={"alpha": ALPHA},
    boundary_conditions={"periodic": {}},
    initial_condition={"type": "sine", "amplitude": 1.0, "frequency": FREQ},
    exact_solution={"type": "sin_exp_decay", "amplitude": 1.0, "frequency": FREQ},
    dimension=1,
    input_dim=2,
    output_dim=1,
    device=device,
    training={
        "num_collocation_points": 1000,
        "num_boundary_points": 100,
        "num_initial_points": 100,
        "loss_weights": {"residual": 1.0, "boundary": 5.0, "initial": 5.0, "smoothness": 0.0},
    },
)

pde = HeatEquation(pde_cfg)
print(f"PDE      : {pde_cfg.name}")
print(f"alpha    : {pde.alpha}")
print(f"Domain   : x in [{X_MIN}, {X_MAX}], t in [{T_MIN}, {T_MAX}]")

PDE      : Heat Equation (notebook demo)
alpha    : 0.1
Domain   : x in [0.0, 1.0], t in [0.0, 1.0]


## 4  Build the Neural Network (Fourier Features Architecture)

**Why Fourier features?** Vanilla MLPs suffer from *spectral bias* — they learn low-frequency
components first and struggle with high-frequency patterns. Fourier feature embeddings lift the
input into a high-dimensional sinusoidal basis, effectively curing this bias.

```
  (x, t) → [sin(B·x), cos(B·x)]  →  MLP  →  u(x,t)
              └── Fourier layer ──┘
```

In [3]:
from pinnrl.neural_networks import PINNModel

# Build a minimal Config-like object that PINNModel expects
model_cfg = ModelConfig(
    input_dim=2,
    hidden_dim=64,
    output_dim=1,
    num_layers=3,
    activation="tanh",
    architecture="fourier",
)
model_cfg.mapping_size = 32
model_cfg.scale = 4.0
model_cfg.device = device

class _Cfg:           # lightweight wrapper so PINNModel(.config) works
    def __init__(self, model_cfg, dev):
        self.model  = model_cfg
        self.device = dev

pinn_config = _Cfg(model_cfg, device)
model = PINNModel(pinn_config, device=device).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Architecture : Fourier Features Network")
print(f"Parameters   : {n_params:,}")
print(f"Device       : {device}")

Architecture : Fourier Features Network
Parameters   : 8,385
Device       : cpu


## 5  The Three-Component Loss

| Component | Equation | Enforces |
|-----------|----------|----------|
| **Residual** | $\mathcal{L}_{\text{pde}} = \frac{1}{N}\sum (u_t - \alpha u_{xx})^2$ | Heat equation is satisfied |
| **Boundary** | $\mathcal{L}_{\text{bc}} = \frac{1}{N}\sum (u(0,t)-u(1,t))^2$ | Periodic BCs match |
| **Initial** | $\mathcal{L}_{\text{ic}} = \frac{1}{N}\sum (u(x,0)-\sin(2\pi x))^2$ | IC is satisfied |

$$\mathcal{L} = w_1\mathcal{L}_{\text{pde}} + w_2\mathcal{L}_{\text{bc}} + w_3\mathcal{L}_{\text{ic}}$$

We delegate the loss computation to `HeatEquation.compute_loss`, which handles auto-diff internally.

## 6  Training Loop (50 Epochs — Transparent, Step-by-Step)

In [4]:
N_EPOCHS    = 50
N_COLL      = 500    # collocation points
LR          = 5e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"total": [], "residual": [], "boundary": [], "initial": []}

print(f"{'Epoch':>6}  {'Total':>10}  {'Residual':>10}  {'Boundary':>10}  {'Initial':>10}")
print("-" * 58)

t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    # ── Sample collocation points uniformly in (x, t) ──────────────────────
    x_coll = torch.rand(N_COLL, 1, device=device) * (X_MAX - X_MIN) + X_MIN
    t_coll = torch.rand(N_COLL, 1, device=device) * (T_MAX - T_MIN) + T_MIN

    # ── Compute all loss terms via the PDE class ────────────────────────────
    losses = pde.compute_loss(model, x_coll, t_coll)

    total_loss = losses["total"]
    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
    optimizer.step()

    # ── Record history ──────────────────────────────────────────────────────
    history["total"].append(losses["total"].item())
    history["residual"].append(losses["residual"].item())
    history["boundary"].append(losses["boundary"].item())
    history["initial"].append(losses["initial"].item())

    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:>6}  {losses['total'].item():>10.4e}  "
              f"{losses['residual'].item():>10.4e}  "
              f"{losses['boundary'].item():>10.4e}  "
              f"{losses['initial'].item():>10.4e}")

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s  |  Final loss: {history['total'][-1]:.4e}")

 Epoch       Total    Residual    Boundary     Initial
----------------------------------------------------------


AttributeError: 'dict' object has no attribute 'num_collocation_points'

## 7  Visualisation

### 7a  Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

epochs_range = range(1, N_EPOCHS + 1)
ax.semilogy(epochs_range, history["total"],    lw=2, label="Total",    color="#1f77b4")
ax.semilogy(epochs_range, history["residual"], lw=1.5, ls="--", label="PDE residual",  color="#ff7f0e")
ax.semilogy(epochs_range, history["boundary"], lw=1.5, ls=":",  label="Boundary",      color="#2ca02c")
ax.semilogy(epochs_range, history["initial"],  lw=1.5, ls="-.", label="Initial cond.", color="#d62728")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss (log scale)", fontsize=12)
ax.set_title("Training Loss — Heat Equation PINN", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(1, N_EPOCHS)
fig.tight_layout()
plt.show()

### 7b  Predicted vs Exact Solution at t = 0.5

In [ ]:
model.eval()

N_PLOT = 200
t_eval = 0.5   # snapshot time

x_plot = torch.linspace(X_MIN, X_MAX, N_PLOT, device=device).unsqueeze(1)
t_plot = torch.full((N_PLOT, 1), t_eval, device=device)

# ── PINN prediction ─────────────────────────────────────────────────────────
with torch.no_grad():
    u_pred = model(torch.cat([x_plot, t_plot], dim=1)).cpu().numpy().ravel()

# ── Exact solution u(x,t) = exp(-alpha*(2*pi*freq)^2*t) * sin(2*pi*freq*x) ─
x_np = x_plot.cpu().numpy().ravel()
decay = np.exp(-ALPHA * (2 * np.pi * FREQ) ** 2 * t_eval)
u_exact = decay * np.sin(2 * np.pi * FREQ * x_np)

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: solution profiles
ax = axes[0]
ax.plot(x_np, u_exact, lw=2.5, color="#1f77b4", label="Exact solution")
ax.plot(x_np, u_pred,  lw=1.8, ls="--", color="#ff7f0e", label="PINN prediction")
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("u(x, t=0.5)", fontsize=12)
ax.set_title(f"Solution at t = {t_eval}", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(X_MIN, X_MAX)

# Right: pointwise absolute error
ax2 = axes[1]
error = np.abs(u_pred - u_exact)
ax2.fill_between(x_np, error, alpha=0.4, color="#d62728")
ax2.plot(x_np, error, lw=1.5, color="#d62728")
ax2.set_xlabel("x", fontsize=12)
ax2.set_ylabel("|u_pred − u_exact|", fontsize=12)
ax2.set_title("Pointwise Absolute Error", fontsize=13, fontweight="bold")
ax2.set_xlim(X_MIN, X_MAX)
ax2.set_ylim(bottom=0)

fig.suptitle("Heat Equation PINN — Fourier Features Network (50 epochs)",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

l2_rel = np.sqrt(np.mean((u_pred - u_exact)**2)) / (np.sqrt(np.mean(u_exact**2)) + 1e-10)
print(f"L2 relative error at t={t_eval}: {l2_rel:.4e}")
print(f"Max absolute error at t={t_eval}: {error.max():.4e}")

### 7c  Space–Time Heat Map

In [ ]:
NX, NT = 100, 100
x_grid = np.linspace(X_MIN, X_MAX, NX)
t_grid = np.linspace(T_MIN, T_MAX, NT)
XX, TT = np.meshgrid(x_grid, t_grid)

xt_flat = torch.tensor(
    np.stack([XX.ravel(), TT.ravel()], axis=1), dtype=torch.float32, device=device
)

model.eval()
with torch.no_grad():
    u_map_pred = model(xt_flat).cpu().numpy().reshape(NT, NX)

decay_map = np.exp(-ALPHA * (2 * np.pi * FREQ) ** 2 * TT)
u_map_exact = decay_map * np.sin(2 * np.pi * FREQ * XX)

vmin = min(u_map_pred.min(), u_map_exact.min())
vmax = max(u_map_pred.max(), u_map_exact.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
kw = dict(extent=[X_MIN, X_MAX, T_MIN, T_MAX], origin="lower",
          aspect="auto", cmap="RdBu_r", vmin=vmin, vmax=vmax)

im0 = axes[0].imshow(u_map_exact, **kw)
axes[0].set_title("Exact u(x,t)", fontweight="bold")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(u_map_pred, **kw)
axes[1].set_title("PINN prediction", fontweight="bold")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im1, ax=axes[1])

err_map = np.abs(u_map_pred - u_map_exact)
im2 = axes[2].imshow(err_map, extent=[X_MIN, X_MAX, T_MIN, T_MAX],
                     origin="lower", aspect="auto", cmap="hot_r")
axes[2].set_title("Absolute error", fontweight="bold")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im2, ax=axes[2])

fig.suptitle("Space–Time Heat Map — Heat Equation PINN",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

## 8  Save the Key Plot

In [ ]:
IMAGES_DIR = os.path.join(os.getcwd(), "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

# Recreate the solution comparison figure and save
fig_save, axes_save = plt.subplots(1, 2, figsize=(13, 4))

ax = axes_save[0]
ax.plot(x_np, u_exact, lw=2.5, color="#1f77b4", label="Exact solution")
ax.plot(x_np, u_pred,  lw=1.8, ls="--", color="#ff7f0e", label="PINN prediction")
ax.set_xlabel("x", fontsize=12); ax.set_ylabel("u(x, t=0.5)", fontsize=12)
ax.set_title(f"Solution at t = {t_eval}", fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.set_xlim(X_MIN, X_MAX)

ax2 = axes_save[1]
ax2.fill_between(x_np, error, alpha=0.4, color="#d62728")
ax2.plot(x_np, error, lw=1.5, color="#d62728")
ax2.set_xlabel("x", fontsize=12); ax2.set_ylabel("|u_pred − u_exact|", fontsize=12)
ax2.set_title("Pointwise Absolute Error", fontsize=13, fontweight="bold")
ax2.set_xlim(X_MIN, X_MAX); ax2.set_ylim(bottom=0)

fig_save.suptitle("Heat Equation PINN — Fourier Features Network (50 epochs)",
                  fontsize=14, fontweight="bold", y=1.02)
fig_save.tight_layout()

save_path = os.path.join(IMAGES_DIR, "01_heat_pinn_solution.png")
fig_save.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close(fig_save)
print(f"Plot saved to: {save_path}")

## 9  What to Try Next

Now that you have a working PINN, here are some experiments to deepen your understanding:

### Increase training
Change `N_EPOCHS = 50` to `500` or `3000`. Run the full training loop and watch the error drop.
You can also use the full trainer:
```bash
python scripts/train.py --pde heat --arch fourier --epochs 3000
```

### Change the physics
- Set `ALPHA = 0.01` (slower diffusion) — the solution will decay more slowly.
- Set `FREQ = 2.0` to add a second harmonic — harder to learn!

### Swap the architecture
Replace `architecture="fourier"` with `"siren"`, `"resnet"`, or `"feedforward"` in the
`ModelConfig` call, then re-run. Notebook **02** will do this comparison systematically.

### Add adaptive weights
The full framework supports automatic loss-weight balancing (`adaptive_weights.enabled: true` in
`config.yaml`). This can significantly improve convergence on stiff PDEs.

### Explore other PDEs
The `pinnrl` library ships with nine PDEs:
`heat`, `wave`, `burgers`, `kdv`, `convection`, `allen_cahn`, `cahn_hilliard`, `black_scholes`, `pendulum`.

---

**Next notebook →** [`02_comparing_architectures.ipynb`](02_comparing_architectures.ipynb)